# visualization_009

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (70).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `indian_equities` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 13 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

# Load the dataset
dat = pd.read_csv('niftyfut_aug.csv')

# Filter rows where the Ticker equals 'NIFTY26SEP24FUT.NFO'
data1 = dat[dat['Ticker'] == 'NIFTY26SEP24FUT.NFO']

# Display the filtered data
data1


In [ ]:
import pandas as pd

# Use a raw string or replace backslashes with forward slashes
data2 = pd.read_csv('niftybees_aug.csv')  # Raw string
# Alternatively:
# data1 = pd.read_csv('ipynb/niftyfut_aug.csv')  # Forward slashes

# Display the DataFrame
data2


In [ ]:
import pandas as pd

# Convert the 'Date_Time' column to datetime format
data2['Date_Time'] = pd.to_datetime(data2['Date_Time'], format="%d-%m-%Y %H:%M")

# Add 59 seconds to each datetime
data2['Date_Time'] = data2['Date_Time'] + pd.Timedelta(seconds=59)

# Convert back to the desired string format if necessary
data2['Date_Time'] = data2['Date_Time'].dt.strftime("%Y-%m-%d %H:%M:%S")

data2


In [ ]:
import pandas as pd

# Convert 'Date_Time' column to datetime in both DataFrames
data1['Date_Time'] = pd.to_datetime(data1['Date_Time'])
data2['Date_Time'] = pd.to_datetime(data2['Date_Time'])

# Merge the two DataFrames on 'Date_Time'
merged_df = pd.merge(data1, data2, on='Date_Time', how='inner', suffixes=('_data1', '_data2'))

# Display the first few rows of the merged DataFrame
merged_df


In [ ]:

# Create the 'data' DataFrame with columns from all merged DataFrames
data = pd.DataFrame({
    'DateTime': merged_df['Date_Time'],
    'niftyfut': merged_df['Close_data1'],
    'niftybees': merged_df['Close_data2'],
})

# Display the resulting DataFrame
data


In [ ]:

# Assuming 'data' is your DataFrame and 'DateTime' is the column name
# Ensure that the 'DateTime' column is in datetime format
data['DateTime'] = pd.to_datetime(data['DateTime'])

# Set 'DateTime' column as the index for data
data.set_index('DateTime', inplace=True)
data

In [ ]:
import pandas as pd

# Filter data to include only trading hours
data = data.between_time('09:15:59', '15:30:59')

# Add a 'date' column for grouping based on index (DateTime)
data['date'] = [x.date() for x in data.index]  # Convert datetime to date

# Group by date and calculate daily open (09:15) and close (15:29) prices
grouped = data.groupby('date')
daily_open = grouped[['niftyfut', 'niftybees']].first()
daily_close = grouped[['niftyfut', 'niftybees']].last()

# Combine daily open and close prices
daily_data = daily_open.rename(columns=lambda x: f"open_{x}")
daily_data = daily_data.join(daily_close.rename(columns=lambda x: f"close_{x}"))

# Ensure that daily_data index is a DatetimeIndex before extracting date
daily_data.index = pd.to_datetime(daily_data.index)

# Add 'date' column after making sure the index is DatetimeIndex
daily_data['date'] = [x.date() for x in daily_data.index]  # Convert datetime to date

# Calculate daily % changes and differences
daily_data['%Change_NiftyFut'] = ((daily_data['close_niftyfut'] - daily_data['open_niftyfut']) / 
                                  daily_data['open_niftyfut']) * 100
daily_data['%Change_NiftyBees'] = ((daily_data['close_niftybees'] - daily_data['open_niftybees']) / 
                                   daily_data['open_niftybees']) * 100
daily_data['Diff'] = daily_data['%Change_NiftyFut'] - daily_data['%Change_NiftyBees']

# Initialize variables
tradebook = []
portfolio_value = 100000  # Example starting portfolio value
active_trade = None

# Extract 'date' from 'DateTime' for both daily and minute-level data
data['date'] = [x.date() for x in data.index]  # Ensure index is a DatetimeIndex

# Minute-level strategy logic
for i in range(len(daily_data) - 1):  # Use daily data for calculating TP
    current_daily = daily_data.iloc[i]
    next_day_data = data[data['date'] == daily_data.iloc[i + 1]['date']]  # Minute-level data for the next day

    # Skip if daily %Change_NiftyFut or %Change_NiftyBees is NaN
    if pd.isna(current_daily['%Change_NiftyFut']) or pd.isna(current_daily['%Change_NiftyBees']):
        continue

    # Long Entry logic at 15:30 based on daily conditions
    if (current_daily['%Change_NiftyFut'] > 0 and current_daily['%Change_NiftyBees'] < 0 and current_daily['Diff'] > 0) or \
       (current_daily['%Change_NiftyFut'] > 0 and current_daily['%Change_NiftyBees'] > 0 and current_daily['Diff'] > 0):
        entry_price = current_daily['close_niftybees']
        tp_price = entry_price * (1 + 0.96*(current_daily['Diff'] / 100))  # Target price for long
        sl_price = entry_price * (1 - 0.01)  # Stop loss (Example: 1% SL)

        # Store the active trade details for long position
        active_trade = {
            'Trade Type': 'Long',
            'Entry Date': current_daily['date'],
            'Entry Time': '15:30',
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
        }

    # Short Entry logic at 15:30 based on daily conditions
    if (current_daily['%Change_NiftyFut'] < 0 and current_daily['%Change_NiftyBees'] > 0 and current_daily['Diff'] < 0) or \
       (current_daily['%Change_NiftyFut'] < 0 and current_daily['%Change_NiftyBees'] < 0 and current_daily['Diff'] < 0):
        entry_price = current_daily['close_niftybees']
        tp_price = entry_price * (0.96 - 0.96*(abs(current_daily['Diff']) / 100))  # Target price for short
        sl_price = entry_price * (0.96 + 0.01)  # Stop loss (Example: 1% SL)

        # Store the active trade details for short position
        active_trade = {
            'Trade Type': 'Short',
            'Entry Date': current_daily['date'],
            'Entry Time': '15:30',
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
        }

    # Exit logic in minute timeframe (15:29 next day if TP or SL not hit)
    if active_trade:
        for minute_index, minute_row in next_day_data.iterrows():
            if minute_row.name.time() == pd.to_datetime('15:29:59').time():
                # Check if TP or SL has been hit
                if (active_trade['Trade Type'] == 'Long' and (minute_row['niftybees'] >= active_trade['TP Price'] or minute_row['niftybees'] <= active_trade['SL Price'])):
                    exit_price = minute_row['niftybees']
                    trade_type = 'TP/SL Hit Long'
                elif (active_trade['Trade Type'] == 'Short' and (minute_row['niftybees'] <= active_trade['TP Price'] or minute_row['niftybees'] >= active_trade['SL Price'])):
                    exit_price = minute_row['niftybees']
                    trade_type = 'TP/SL Hit Short'
                else:
                    exit_price = minute_row['niftybees']  # Regular exit at 15:29
                    trade_type = f"Exit at 15:29 {active_trade['Trade Type']}"

                # Record trade and reset active trade
                active_trade['Exit Date'] = minute_row.name.date()
                active_trade['Exit Time'] = minute_row.name.time()
                active_trade['Exit Price'] = exit_price
                active_trade['Trade Result'] = 'Profit' if active_trade['Trade Type'] == 'Long' and exit_price > active_trade['Entry Price'] else 'Loss'
                active_trade['PnL'] = (exit_price - active_trade['Entry Price']) if active_trade['Trade Type'] == 'Long' else (active_trade['Entry Price'] - exit_price)
                active_trade['%Change_NiftyFut'] = current_daily['%Change_NiftyFut']
                active_trade['%Change_NiftyBees'] = current_daily['%Change_NiftyBees']
                active_trade['Diff'] = current_daily['Diff']
                active_trade['Exit Reason'] = trade_type

                tradebook.append(active_trade)
                active_trade = None  # Reset active trade
                break  # Exit the loop since the trade is closed

# Convert tradebook to DataFrame
tradebook_df = pd.DataFrame(tradebook)

# Output results
if not tradebook_df.empty:
    print(tradebook_df)
    print(f"Total PnL: {tradebook_df['PnL'].sum():.2f}")
else:
    print("No trades executed.")


In [ ]:
tradebook_df

In [ ]:
import pandas as pd

# Filter data to include only trading hours
data = data.between_time('09:15:59', '15:30:59')

# Add a 'date' column for grouping based on index (DateTime)
data['date'] = [x.date() for x in data.index]  # Convert datetime to date

# Group by date and calculate daily open (09:15) and close (15:29) prices
grouped = data.groupby('date')
daily_open = grouped[['niftyfut', 'niftybees']].first()
daily_close = grouped[['niftyfut', 'niftybees']].last()

# Combine daily open and close prices
daily_data = daily_open.rename(columns=lambda x: f"open_{x}")
daily_data = daily_data.join(daily_close.rename(columns=lambda x: f"close_{x}"))

# Ensure that daily_data index is a DatetimeIndex before extracting date
daily_data.index = pd.to_datetime(daily_data.index)

# Add 'date' column after making sure the index is DatetimeIndex
daily_data['date'] = [x.date() for x in daily_data.index]  # Convert datetime to date

# Calculate daily % changes and differences
daily_data['%Change_NiftyFut'] = ((daily_data['close_niftyfut'] - daily_data['open_niftyfut']) / 
                                  daily_data['open_niftyfut']) * 100
daily_data['%Change_NiftyBees'] = ((daily_data['close_niftybees'] - daily_data['open_niftybees']) / 
                                   daily_data['open_niftybees']) * 100
daily_data['Diff'] = daily_data['%Change_NiftyFut'] - daily_data['%Change_NiftyBees']

# Initialize variables
tradebook = []
portfolio_value = 100000  # Example starting portfolio value
initial_portfolio_value = portfolio_value
active_trade = None
brokerage_fee = 0.001  # 0.1% brokerage fee

# Extract 'date' from 'DateTime' for both daily and minute-level data
data['date'] = [x.date() for x in data.index]  # Ensure index is a DatetimeIndex

# Initialize lists for max drawdown and relative drawdown
max_drawdown = 0
relative_drawdown = 0
peak_value = portfolio_value

# Minute-level strategy logic
for i in range(len(daily_data) - 1):  # Use daily data for calculating TP
    current_daily = daily_data.iloc[i]
    next_day_data = data[data['date'] == daily_data.iloc[i + 1]['date']]  # Minute-level data for the next day

    # Skip if daily %Change_NiftyFut or %Change_NiftyBees is NaN
    if pd.isna(current_daily['%Change_NiftyFut']) or pd.isna(current_daily['%Change_NiftyBees']):
        continue

    # Long Entry logic at 15:30 based on daily conditions
    if (current_daily['%Change_NiftyFut'] > 0 and current_daily['%Change_NiftyBees'] < 0 and current_daily['Diff'] > 0) or \
       (current_daily['%Change_NiftyFut'] > 0 and current_daily['%Change_NiftyBees'] > 0 and current_daily['Diff'] > 0):
        entry_price = current_daily['close_niftybees']
        tp_price = entry_price * (1 + 0.98*(current_daily['Diff'] / 100))  # Target price for long
        sl_price = entry_price * (1 - 0.01)  # Stop loss (Example: 1% SL)

        # Store the active trade details for long position
        active_trade = {
            'Trade Type': 'Long',
            'Entry Date': current_daily['date'],
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Position Size': portfolio_value / entry_price  # Position size based on available capital
        }

    # Short Entry logic at 15:30 based on daily conditions
    if (current_daily['%Change_NiftyFut'] < 0 and current_daily['%Change_NiftyBees'] > 0 and current_daily['Diff'] < 0) or \
       (current_daily['%Change_NiftyFut'] < 0 and current_daily['%Change_NiftyBees'] < 0 and current_daily['Diff'] < 0):
        entry_price = current_daily['close_niftybees']
        tp_price = entry_price * (1 - 0.98*(abs(current_daily['Diff']) / 100))  # Target price for short
        sl_price = entry_price * (1 + 0.01)  # Stop loss (Example: 1% SL)

        # Store the active trade details for short position
        active_trade = {
            'Trade Type': 'Short',
            'Entry Date': current_daily['date'],
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Position Size': portfolio_value / entry_price  # Position size based on available capital
        }

    # Exit logic in minute timeframe (15:29 next day if TP or SL not hit)
    if active_trade:
        for minute_index, minute_row in next_day_data.iterrows():
            if minute_row.name.time() == pd.to_datetime('15:29:59').time():
                # Check if TP or SL has been hit
                if (active_trade['Trade Type'] == 'Long' and (minute_row['niftybees'] >= active_trade['TP Price'] or minute_row['niftybees'] <= active_trade['SL Price'])):
                    exit_price = minute_row['niftybees']
                    trade_type = 'TP/SL Hit Long'
                elif (active_trade['Trade Type'] == 'Short' and (minute_row['niftybees'] <= active_trade['TP Price'] or minute_row['niftybees'] >= active_trade['SL Price'])):
                    exit_price = minute_row['niftybees']
                    trade_type = 'TP/SL Hit Short'
                else:
                    exit_price = minute_row['niftybees']  # Regular exit at 15:29
                    trade_type = f"Exit at 15:29 {active_trade['Trade Type']}"

                # Calculate the trade PnL
                trade_pnl = (exit_price - active_trade['Entry Price']) * active_trade['Position Size'] if active_trade['Trade Type'] == 'Long' else (active_trade['Entry Price'] - exit_price) * active_trade['Position Size']
                
                # Subtract brokerage fee
                trade_pnl -= 100000 * brokerage_fee

                # Update portfolio value
                portfolio_value += trade_pnl

                # Track the max drawdown and relative drawdown
                peak_value = max(peak_value, portfolio_value)
                drawdown = peak_value - portfolio_value
                max_drawdown = max(max_drawdown, drawdown)
                relative_drawdown = max(relative_drawdown, drawdown / peak_value)

                # Record trade and reset active trade
                active_trade['Exit Date'] = minute_row.name.date()
                active_trade['Exit Price'] = exit_price
                active_trade['Trade Result'] = 'Profit' if trade_pnl > 0 else 'Loss'
                active_trade['PnL'] = trade_pnl
                active_trade['Exit Reason'] = trade_type

                tradebook.append(active_trade)
                active_trade = None  # Reset active trade
                break  # Exit the loop since the trade is closed

# Convert tradebook to DataFrame
tradebook_df = pd.DataFrame(tradebook)

# Output results
if not tradebook_df.empty:
    print(tradebook_df)
    print(f"Total PnL: {tradebook_df['PnL'].sum():.2f}")
    print(f"Max Drawdown: {max_drawdown:.2f}")
    print(f"Relative Drawdown: {relative_drawdown:.2f}")
else:
    print("No trades executed.")


In [ ]:
t = tradebook_df.to_csv('tradeboook_df.csv', index=False)
t

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pandas.plotting import lag_plot
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.backends.backend_pdf import PdfPages

# Assuming tradebook_df contains the necessary columns: 'Exit Date', 'PnL'
# Make sure the data types are correct, especially for the 'Exit Date' column which should be datetime type.
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])

# Calculate cumulative PnL
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()

# Calculate Drawdown and Relative Drawdown
tradebook_df['Peak'] = tradebook_df['Cumulative PnL'].cummax()
tradebook_df['Drawdown'] = tradebook_df['Cumulative PnL'] - tradebook_df['Peak']
tradebook_df['Relative Drawdown'] = tradebook_df['Drawdown'] / tradebook_df['Peak'] * 100

# Start PDF saving process
with PdfPages('combined_charts.pdf') as pdf:

    # 1. Plot Cumulative Profit Over Time
    plt.figure(figsize=(10, 6))
    plt.plot(tradebook_df['Exit Date'], tradebook_df['Cumulative PnL'], label='Cumulative PnL', color='blue')
    plt.title('Cumulative Profit Over Time')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Profit (₹)')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 2. Plot Max Drawdown and Relative Drawdown Over Time
    plt.figure(figsize=(10, 6))
    plt.plot(tradebook_df['Exit Date'], tradebook_df['Drawdown'], label='Max Drawdown', color='red')
    plt.plot(tradebook_df['Exit Date'], tradebook_df['Relative Drawdown'], label='Relative Drawdown', color='green')
    plt.title('Max Drawdown and Relative Drawdown Over Time')
    plt.xlabel('Date')
    plt.ylabel('Drawdown')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 3. Plot PnL Distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(tradebook_df['PnL'], bins=30, kde=True, color='purple')
    plt.title('PnL Distribution')
    plt.xlabel('PnL (₹)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 4. Plot Profit Distribution by Weekdays
    tradebook_df['Weekday'] = tradebook_df['Exit Date'].dt.weekday
    weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    tradebook_df['Weekday'] = tradebook_df['Weekday'].apply(lambda x: weekday_names[x])

    plt.figure(figsize=(12, 6))
    sns.boxplot(x='Weekday', y='PnL', data=tradebook_df, order=weekday_names, palette='Set2')
    plt.title('Profit Distribution by Weekdays')
    plt.xlabel('Weekday')
    plt.ylabel('Profit/Loss (₹)')
    plt.grid(True)
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 5. Scatter Plot: Nifty Futures vs Nifty Bees (using some hypothetical data)
    # Assuming you have a DataFrame 'data' containing 'niftyfut' and 'niftybees' columns
    plt.figure(figsize=(10, 6))
    plt.scatter(data['niftyfut'], data['niftybees'], color='purple')
    plt.title('Nifty Futures vs Nifty Bees (Scatter Plot)')
    plt.xlabel('Nifty Futures')
    plt.ylabel('Nifty Bees')
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 6. Heatmap: Correlation between Nifty Futures and Nifty Bees
    corr = data[['niftyfut', 'niftybees']].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap between Nifty Futures and Nifty Bees')
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 7. Pair Plot: Nifty Futures and Nifty Bees
    sns.pairplot(data[['niftyfut', 'niftybees']], diag_kind='kde', height=3)
    plt.suptitle('Pair Plot of Nifty Futures and Nifty Bees', y=1.02)
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 8. Lag Plot: Detect Autocorrelation in Nifty Futures
    plt.figure(figsize=(10, 6))
    lag_plot(data['niftyfut'])
    plt.title('Lag Plot of Nifty Futures')
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

    # 9. Time Series Decomposition (using hypothetical time series data)
    result = seasonal_decompose(data.set_index('date')['niftyfut'], model='additive', period=1)
    result.plot()
    plt.suptitle('Time Series Decomposition')
    plt.tight_layout()
    pdf.savefig()  # Save the current figure to PDF
    plt.close()

# All figures are now saved into 'combined_charts.pdf'
